In [1]:
import sqlalchemy

In [2]:
# ============================================================
# IMPORTS — CARGA MYSQL
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
from getpass import getpass

import sqlalchemy

from sqlalchemy import (
    create_engine,
    text
)


print("=" * 70)
print("AMBIENTE PYTHON — MYSQL")
print("=" * 70)

print(
    "\nPandas:",
    pd.__version__
)

print(
    "SQLAlchemy:",
    sqlalchemy.__version__
)

AMBIENTE PYTHON — MYSQL

Pandas: 2.3.3
SQLAlchemy: 2.0.43


In [3]:
# ============================================================
# LOCALIZAR O PROJETO E A BASE ANUAL FINAL
# ============================================================

caminho_atual = Path.cwd()


candidatos_raiz = [
    caminho_atual,
    *caminho_atual.parents
]


RAIZ_PROJETO = next(
    (
        caminho
        for caminho in candidatos_raiz
        if (
            (caminho / "data").exists()
            and
            (caminho / "notebooks").exists()
        )
    ),
    None
)


if RAIZ_PROJETO is None:

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


ARQUIVO_FATO_AGROAMBIENTAL = (
    RAIZ_PROJETO
    /
    "data"
    /
    "databases_curated"
    /
    "integracao_agroambiental"
    /
    "base_agroambiental_final_soja_centro_oeste_sul_2019_2024.csv"
)


print("=" * 70)
print("ARQUIVO DE ORIGEM")
print("=" * 70)


print(
    "\nRaiz:",
    RAIZ_PROJETO
)


print(
    "\nArquivo:"
)

print(
    ARQUIVO_FATO_AGROAMBIENTAL
)


print(
    "\nExiste:",
    ARQUIVO_FATO_AGROAMBIENTAL.exists()
)

ARQUIVO DE ORIGEM

Raiz: C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao

Arquivo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\integracao_agroambiental\base_agroambiental_final_soja_centro_oeste_sul_2019_2024.csv

Existe: True


In [4]:
# ============================================================
# CARREGAR BASE ANUAL FINAL
# ============================================================

fato_agroambiental = pd.read_csv(
    ARQUIVO_FATO_AGROAMBIENTAL,
    dtype={
        "codigo_ibge": "string"
    }
)


print("=" * 70)
print("BASE ANUAL PARA CARGA MYSQL")
print("=" * 70)


print(
    "\nDimensão:",
    fato_agroambiental.shape
)


print(
    "Municípios:",
    fato_agroambiental[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "Anos:"
)

print(
    sorted(
        fato_agroambiental[
            "ano"
        ]
        .unique()
        .tolist()
    )
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    fato_agroambiental
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print(
    "\nNULL totais:"
)

print(
    fato_agroambiental
    .isna()
    .sum()
    .sum()
)


print(
    "\nTipo codigo_ibge:"
)

print(
    fato_agroambiental[
        "codigo_ibge"
    ]
    .dtype
)


print(
    "\nComprimento codigo_ibge:"
)

display(
    fato_agroambiental[
        "codigo_ibge"
    ]
    .str.len()
    .value_counts()
    .sort_index()
)

BASE ANUAL PARA CARGA MYSQL

Dimensão: (8674, 134)
Municípios: 1505
Anos:
[2019, 2020, 2021, 2022, 2023, 2024]

Duplicatas codigo_ibge + ano:
0

NULL totais:
789

Tipo codigo_ibge:
string

Comprimento codigo_ibge:


codigo_ibge
7    8674
Name: count, dtype: Int64

In [5]:
# ============================================================
# INVENTÁRIO DE TIPOS — BASE ANUAL
# ============================================================

resumo_tipos = (
    fato_agroambiental
    .dtypes
    .astype(str)
    .value_counts()
    .rename_axis("dtype")
    .reset_index(name="quantidade_colunas")
)


print("=" * 70)
print("TIPOS DAS COLUNAS")
print("=" * 70)

display(
    resumo_tipos
)


# ------------------------------------------------------------
# Listar as colunas por tipo
# ------------------------------------------------------------

for dtype in fato_agroambiental.dtypes.unique():

    colunas_tipo = (
        fato_agroambiental
        .columns[
            fato_agroambiental.dtypes == dtype
        ]
        .tolist()
    )

    print("\n" + "-" * 70)
    print("TIPO:", dtype)
    print("QUANTIDADE:", len(colunas_tipo))
    print("-" * 70)

    for coluna in colunas_tipo:

        print(
            "->",
            coluna
        )

TIPOS DAS COLUNAS


,dtype,quantidade_colunas
0,float64,94
1,object,27
2,int64,10
3,bool,2
4,string,1



----------------------------------------------------------------------
TIPO: string
QUANTIDADE: 1
----------------------------------------------------------------------
-> codigo_ibge

----------------------------------------------------------------------
TIPO: object
QUANTIDADE: 27
----------------------------------------------------------------------
-> municipio
-> uf
-> regiao
-> cultura
-> status_dado
-> origem_precipitacao
-> origem_temperatura
-> origem_umidade
-> qualidade_espacial_precipitacao
-> qualidade_espacial_temperatura
-> qualidade_espacial_umidade
-> tipo_representacao_climatica
-> qualidade_climatica_geral
-> biomas_presentes
-> faixa_diferenca_area_ibge
-> biomas_presentes_seeg
-> status_dado_seeg
-> fonte_seeg
-> setor_emissao
-> categoria_emissao
-> subcategoria_emissao
-> gas_origem
-> metrica_co2e
-> fonte_brluc
-> versao_fonte
-> classe_brluc_t0
-> classe_brluc_t1

----------------------------------------------------------------------
TIPO: int64
QUANTIDADE: 1

In [6]:
# ============================================================
# AUDITORIA DAS COLUNAS TEXTUAIS
# ============================================================

colunas_textuais = (
    fato_agroambiental
    .select_dtypes(
        include=[
            "object",
            "string"
        ]
    )
    .columns
    .tolist()
)


auditoria_textos = []


for coluna in colunas_textuais:

    serie = (
        fato_agroambiental[
            coluna
        ]
        .dropna()
        .astype(str)
    )


    auditoria_textos.append(
        {
            "coluna":
                coluna,

            "dtype_python":
                str(
                    fato_agroambiental[
                        coluna
                    ].dtype
                ),

            "n_nao_null":
                len(
                    serie
                ),

            "n_distintos":
                serie.nunique(),

            "comprimento_maximo":
                (
                    serie
                    .str.len()
                    .max()
                    if len(serie) > 0
                    else 0
                ),

            "exemplo":
                (
                    serie.iloc[0]
                    if len(serie) > 0
                    else None
                )
        }
    )


auditoria_textos = (
    pd.DataFrame(
        auditoria_textos
    )
    .sort_values(
        by="comprimento_maximo",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("=" * 70)
print("AUDITORIA DAS COLUNAS TEXTUAIS")
print("=" * 70)


display(
    auditoria_textos
)

AUDITORIA DAS COLUNAS TEXTUAIS


,coluna,dtype_python,n_nao_null,n_distintos,comprimento_maximo,exemplo
0,classe_brluc_t1,object,8674,1,58,"Cropland, temporary, generic, average tillage ..."
1,classe_brluc_t0,object,8674,1,58,"Cropland, temporary, generic, average tillage ..."
2,municipio,object,8674,1482,32,Brasília
3,biomas_presentes_seeg,object,8674,10,29,Cerrado
4,biomas_presentes,object,8674,10,29,Cerrado
5,tipo_representacao_climatica,object,8674,3,24,observado_tres_variaveis
6,status_dado_seeg,object,8674,2,19,completo
7,subcategoria_emissao,object,8674,1,18,Resíduos agrícolas
8,faixa_diferenca_area_ibge,object,8674,3,17,ate_0_1_pct
9,categoria_emissao,object,8674,1,15,Solos manejados


In [8]:
# ============================================================
# VALIDAR DRIVER MYSQL
# ============================================================

import pymysql


print("=" * 70)
print("PYMYSQL")
print("=" * 70)

print(
    "\nVersão:",
    pymysql.__version__
)

PYMYSQL

Versão: 1.4.6


In [9]:
# ============================================================
# CONEXÃO PYTHON → MYSQL
# ============================================================

from getpass import getpass

from sqlalchemy import (
    create_engine,
    text,
    URL
)


USUARIO_MYSQL = input(
    "Usuário MySQL: "
)


SENHA_MYSQL = getpass(
    "Senha MySQL: "
)


url_mysql = URL.create(
    drivername="mysql+pymysql",
    username=USUARIO_MYSQL,
    password=SENHA_MYSQL,
    host="localhost",
    port=3306,
    database="agroesg_analytics"
)


engine = create_engine(
    url_mysql,
    pool_pre_ping=True
)


print("=" * 70)
print("TESTE DE CONEXÃO MYSQL")
print("=" * 70)


with engine.connect() as conexao:

    versao_mysql = (
        conexao.execute(
            text(
                "SELECT VERSION();"
            )
        )
        .scalar()
    )


    banco_atual = (
        conexao.execute(
            text(
                "SELECT DATABASE();"
            )
        )
        .scalar()
    )


print(
    "\nVersão MySQL:",
    versao_mysql
)


print(
    "Banco atual:",
    banco_atual
)

Usuário MySQL:  root
Senha MySQL:  ········


TESTE DE CONEXÃO MYSQL

Versão MySQL: 8.0.46
Banco atual: agroesg_analytics


In [10]:
# ============================================================
# TIPOS SQL — FATO AGROAMBIENTAL ANUAL
# ============================================================

from sqlalchemy.dialects.mysql import (
    CHAR,
    VARCHAR,
    SMALLINT,
    DOUBLE,
    BOOLEAN
)


TIPOS_SQL_FATO = {}


# ------------------------------------------------------------
# Identificadores
# ------------------------------------------------------------

TIPOS_SQL_FATO[
    "codigo_ibge"
] = CHAR(7)


TIPOS_SQL_FATO[
    "uf"
] = CHAR(2)


# ------------------------------------------------------------
# Colunas textuais
#
# O maior texto encontrado possui 58 caracteres.
# VARCHAR(100) oferece margem sem exagero.
# ------------------------------------------------------------

for coluna in fato_agroambiental.select_dtypes(
    include=[
        "object",
        "string"
    ]
).columns:

    if coluna not in [
        "codigo_ibge",
        "uf"
    ]:

        TIPOS_SQL_FATO[
            coluna
        ] = VARCHAR(100)


# ------------------------------------------------------------
# Inteiros
#
# Todos os inteiros desta base possuem valores compatíveis
# com SMALLINT: anos, scores e contagens pequenas.
# ------------------------------------------------------------

for coluna in fato_agroambiental.select_dtypes(
    include=[
        "int64"
    ]
).columns:

    TIPOS_SQL_FATO[
        coluna
    ] = SMALLINT()


# ------------------------------------------------------------
# Decimais
# ------------------------------------------------------------

for coluna in fato_agroambiental.select_dtypes(
    include=[
        "float64"
    ]
).columns:

    TIPOS_SQL_FATO[
        coluna
    ] = DOUBLE()


# ------------------------------------------------------------
# Booleanos
# ------------------------------------------------------------

for coluna in fato_agroambiental.select_dtypes(
    include=[
        "bool"
    ]
).columns:

    TIPOS_SQL_FATO[
        coluna
    ] = BOOLEAN()


print("=" * 70)
print("MAPEAMENTO SQL")
print("=" * 70)


print(
    "\nColunas da base:",
    len(
        fato_agroambiental.columns
    )
)


print(
    "Colunas com tipo SQL definido:",
    len(
        TIPOS_SQL_FATO
    )
)


colunas_sem_tipo = (
    set(
        fato_agroambiental.columns
    )
    -
    set(
        TIPOS_SQL_FATO.keys()
    )
)


print(
    "Colunas sem tipo:",
    len(
        colunas_sem_tipo
    )
)


if colunas_sem_tipo:

    print(
        colunas_sem_tipo
    )

MAPEAMENTO SQL

Colunas da base: 134
Colunas com tipo SQL definido: 134
Colunas sem tipo: 0


In [11]:
# ============================================================
# VERIFICAR EXISTÊNCIA DA TABELA
# ============================================================

from sqlalchemy import inspect


NOME_TABELA_FATO = (
    "fato_agroambiental_anual"
)


inspector = inspect(
    engine
)


tabela_ja_existe = (
    inspector.has_table(
        NOME_TABELA_FATO
    )
)


print("=" * 70)
print("VERIFICAÇÃO DA TABELA")
print("=" * 70)


print(
    "\nTabela:",
    NOME_TABELA_FATO
)


print(
    "Já existe:",
    tabela_ja_existe
)

VERIFICAÇÃO DA TABELA

Tabela: fato_agroambiental_anual
Já existe: False


In [12]:
# ============================================================
# CARGA — FATO AGROAMBIENTAL ANUAL
# ============================================================

if tabela_ja_existe:

    raise RuntimeError(
        "A tabela fato_agroambiental_anual já existe. "
        "Carga interrompida para evitar sobrescrita."
    )


print("=" * 70)
print("INICIANDO CARGA MYSQL")
print("=" * 70)


fato_agroambiental.to_sql(
    name=NOME_TABELA_FATO,
    con=engine,
    if_exists="fail",
    index=False,
    dtype=TIPOS_SQL_FATO,
    chunksize=200,
    method="multi"
)


print(
    "\nCarga concluída."
)

INICIANDO CARGA MYSQL

Carga concluída.


In [13]:
# ============================================================
# VALIDAÇÃO — CSV × MYSQL
# ============================================================

# ------------------------------------------------------------
# Construir expressão para contar todos os NULL do MySQL
# ------------------------------------------------------------

expressoes_null = []


for coluna in fato_agroambiental.columns:

    coluna_escapada = (
        coluna
        .replace(
            "`",
            "``"
        )
    )

    expressoes_null.append(
        f"""
        SUM(
            CASE
                WHEN `{coluna_escapada}` IS NULL
                THEN 1
                ELSE 0
            END
        )
        """
    )


expressao_total_null = (
    " + ".join(
        expressoes_null
    )
)


with engine.connect() as conexao:

    # --------------------------------------------------------
    # Quantidade total de linhas
    # --------------------------------------------------------

    linhas_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_FATO}`;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Municípios distintos
    # --------------------------------------------------------

    municipios_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(
                    DISTINCT codigo_ibge
                )
                FROM `{NOME_TABELA_FATO}`;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Intervalo de anos
    # --------------------------------------------------------

    resultado_anos = (
        conexao.execute(
            text(
                f"""
                SELECT
                    MIN(ano),
                    MAX(ano),
                    COUNT(DISTINCT ano)
                FROM `{NOME_TABELA_FATO}`;
                """
            )
        )
        .one()
    )


    # --------------------------------------------------------
    # Duplicatas da chave lógica
    # --------------------------------------------------------

    duplicatas_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM (
                    SELECT
                        codigo_ibge,
                        ano,
                        COUNT(*) AS quantidade
                    FROM `{NOME_TABELA_FATO}`
                    GROUP BY
                        codigo_ibge,
                        ano
                    HAVING COUNT(*) > 1
                ) AS duplicatas;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # NULL nas futuras colunas da PK
    # --------------------------------------------------------

    null_chave_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_FATO}`
                WHERE
                    codigo_ibge IS NULL
                    OR ano IS NULL;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Total de células NULL
    # --------------------------------------------------------

    total_null_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT
                    {expressao_total_null}
                FROM `{NOME_TABELA_FATO}`;
                """
            )
        )
        .scalar()
    )


# ============================================================
# COMPARAÇÃO
# ============================================================

total_null_csv = (
    fato_agroambiental
    .isna()
    .sum()
    .sum()
)


print("=" * 70)
print("VALIDAÇÃO CSV × MYSQL")
print("=" * 70)


print(
    "\nLinhas CSV:",
    len(
        fato_agroambiental
    )
)

print(
    "Linhas MySQL:",
    linhas_mysql
)


print(
    "\nMunicípios CSV:",
    fato_agroambiental[
        "codigo_ibge"
    ]
    .nunique()
)

print(
    "Municípios MySQL:",
    municipios_mysql
)


print(
    "\nAno mínimo MySQL:",
    resultado_anos[0]
)

print(
    "Ano máximo MySQL:",
    resultado_anos[1]
)

print(
    "Quantidade de anos:",
    resultado_anos[2]
)


print(
    "\nDuplicatas codigo_ibge + ano:",
    duplicatas_mysql
)


print(
    "NULL na futura chave primária:",
    null_chave_mysql
)


print(
    "\nNULL totais CSV:",
    total_null_csv
)

print(
    "NULL totais MySQL:",
    total_null_mysql
)


print(
    "\nNULL preservados corretamente:",
    total_null_csv
    ==
    total_null_mysql
)

VALIDAÇÃO CSV × MYSQL

Linhas CSV: 8674
Linhas MySQL: 8674

Municípios CSV: 1505
Municípios MySQL: 1505

Ano mínimo MySQL: 2019
Ano máximo MySQL: 2024
Quantidade de anos: 6

Duplicatas codigo_ibge + ano: 0
NULL na futura chave primária: 0

NULL totais CSV: 789
NULL totais MySQL: 789

NULL preservados corretamente: True


In [14]:
# ============================================================
# PRIMARY KEY E ÍNDICES — FATO AGROAMBIENTAL ANUAL
# ============================================================

from sqlalchemy import inspect


# ------------------------------------------------------------
# Verificar PK atual
# ------------------------------------------------------------

inspector = inspect(
    engine
)


pk_atual = (
    inspector
    .get_pk_constraint(
        NOME_TABELA_FATO
    )
)


colunas_pk_atual = (
    pk_atual
    .get(
        "constrained_columns"
    )
    or []
)


print("=" * 70)
print("CHAVE PRIMÁRIA E ÍNDICES")
print("=" * 70)


print(
    "\nPK antes:",
    colunas_pk_atual
)


# ------------------------------------------------------------
# Criar PK somente se ainda não existir
# ------------------------------------------------------------

if not colunas_pk_atual:

    with engine.begin() as conexao:

        conexao.execute(
            text(
                f"""
                ALTER TABLE `{NOME_TABELA_FATO}`

                MODIFY `codigo_ibge`
                    CHAR(7) NOT NULL,

                MODIFY `ano`
                    SMALLINT NOT NULL,

                ADD CONSTRAINT
                    `pk_fato_agroambiental_anual`

                PRIMARY KEY (
                    `codigo_ibge`,
                    `ano`
                );
                """
            )
        )

    print(
        "Primary Key criada."
    )

else:

    print(
        "Primary Key já existe."
    )


# ------------------------------------------------------------
# Atualizar inspector após ALTER TABLE
# ------------------------------------------------------------

inspector = inspect(
    engine
)


indices_existentes = {
    indice[
        "name"
    ]
    for indice in inspector.get_indexes(
        NOME_TABELA_FATO
    )
}


# ------------------------------------------------------------
# Índice por ano
# ------------------------------------------------------------

if "idx_fato_agroambiental_ano" not in indices_existentes:

    with engine.begin() as conexao:

        conexao.execute(
            text(
                f"""
                CREATE INDEX
                    `idx_fato_agroambiental_ano`

                ON `{NOME_TABELA_FATO}`
                    (`ano`);
                """
            )
        )


# ------------------------------------------------------------
# Índice por UF
# ------------------------------------------------------------

if "idx_fato_agroambiental_uf" not in indices_existentes:

    with engine.begin() as conexao:

        conexao.execute(
            text(
                f"""
                CREATE INDEX
                    `idx_fato_agroambiental_uf`

                ON `{NOME_TABELA_FATO}`
                    (`uf`);
                """
            )
        )


# ------------------------------------------------------------
# Índice por região
# ------------------------------------------------------------

if "idx_fato_agroambiental_regiao" not in indices_existentes:

    with engine.begin() as conexao:

        conexao.execute(
            text(
                f"""
                CREATE INDEX
                    `idx_fato_agroambiental_regiao`

                ON `{NOME_TABELA_FATO}`
                    (`regiao`);
                """
            )
        )


print(
    "\nÍndices concluídos."
)

CHAVE PRIMÁRIA E ÍNDICES

PK antes: []
Primary Key criada.

Índices concluídos.


In [15]:
# ============================================================
# AUDITORIA DA ESTRUTURA MYSQL
# ============================================================

inspector = inspect(
    engine
)


pk_final = (
    inspector
    .get_pk_constraint(
        NOME_TABELA_FATO
    )
)


indices_finais = (
    inspector
    .get_indexes(
        NOME_TABELA_FATO
    )
)


print("=" * 70)
print("ESTRUTURA MYSQL — FATO AGROAMBIENTAL ANUAL")
print("=" * 70)


print(
    "\nPrimary Key:"
)

print(
    pk_final
)


print(
    "\nÍndices:"
)


for indice in indices_finais:

    print(
        "->",
        indice[
            "name"
        ],
        "|",
        indice[
            "column_names"
        ]
    )


# ------------------------------------------------------------
# Conferir definição das duas colunas-chave
# ------------------------------------------------------------

consulta_chaves = """
SELECT
    COLUMN_NAME,
    COLUMN_TYPE,
    IS_NULLABLE,
    COLUMN_KEY
FROM information_schema.COLUMNS
WHERE
    TABLE_SCHEMA = 'agroesg_analytics'
    AND TABLE_NAME = 'fato_agroambiental_anual'
    AND COLUMN_NAME IN (
        'codigo_ibge',
        'ano'
    )
ORDER BY
    ORDINAL_POSITION;
"""


estrutura_chaves = pd.read_sql(
    text(
        consulta_chaves
    ),
    engine
)


print(
    "\nEstrutura das chaves:"
)


display(
    estrutura_chaves
)


# ------------------------------------------------------------
# Amostra da tabela já no MySQL
# ------------------------------------------------------------

amostra_mysql = pd.read_sql(
    text(
        """
        SELECT
            codigo_ibge,
            municipio,
            uf,
            regiao,
            ano,
            cultura,
            area_plantada_ha,
            quantidade_produzida_t,
            precipitacao_anual_mm,
            carbono_solo_t_ha
        FROM fato_agroambiental_anual
        ORDER BY
            codigo_ibge,
            ano
        LIMIT 10;
        """
    ),
    engine
)


print(
    "\nAmostra:"
)


display(
    amostra_mysql
)

ESTRUTURA MYSQL — FATO AGROAMBIENTAL ANUAL

Primary Key:
{'constrained_columns': ['codigo_ibge', 'ano'], 'name': None}

Índices:
-> idx_fato_agroambiental_ano | ['ano']
-> idx_fato_agroambiental_regiao | ['regiao']
-> idx_fato_agroambiental_uf | ['uf']

Estrutura das chaves:


,COLUMN_NAME,COLUMN_TYPE,IS_NULLABLE,COLUMN_KEY
0,codigo_ibge,char(7),NO,PRI
1,ano,smallint,NO,PRI



Amostra:


,codigo_ibge,municipio,uf,regiao,ano,cultura,area_plantada_ha,quantidade_produzida_t,precipitacao_anual_mm,carbono_solo_t_ha
0,4100103,Abatiá,PR,Sul,2019,soja,10570.0,36784.0,994.780491,55.173093
1,4100103,Abatiá,PR,Sul,2020,soja,10470.0,36435.0,907.189175,55.169301
2,4100103,Abatiá,PR,Sul,2021,soja,10220.0,27727.0,1219.103697,55.039162
3,4100103,Abatiá,PR,Sul,2022,soja,10470.0,37064.0,1501.673064,55.024834
4,4100103,Abatiá,PR,Sul,2023,soja,9960.0,35856.0,1697.612945,55.025820
5,4100103,Abatiá,PR,Sul,2024,soja,9600.0,34560.0,999.928713,55.022820
6,4100202,Adrianópolis,PR,Sul,2023,soja,20.0,70.0,2018.039145,56.421824
7,4100202,Adrianópolis,PR,Sul,2024,soja,18.0,61.0,1653.049405,56.418058
8,4100301,Agudos do Sul,PR,Sul,2019,soja,1450.0,5365.0,1354.124036,66.305396
9,4100301,Agudos do Sul,PR,Sul,2020,soja,1840.0,7084.0,1185.581568,66.255196


In [16]:
# ============================================================
# LOCALIZAR E CARREGAR — MART PRIORIZAÇÃO MUNICIPAL
# ============================================================

ARQUIVO_MART_PRIORIZACAO = (
    RAIZ_PROJETO
    /
    "data"
    /
    "databases_curated"
    /
    "priorizacao_agroambiental"
    /
    "base_modelo_priorizacao_agroambiental_soja_centro_oeste_sul_municipio.csv"
)


print("=" * 70)
print("ARQUIVO — MART PRIORIZAÇÃO")
print("=" * 70)


print(
    "\nArquivo:"
)

print(
    ARQUIVO_MART_PRIORIZACAO
)


print(
    "\nExiste:",
    ARQUIVO_MART_PRIORIZACAO.exists()
)


# ============================================================
# CARREGAR
# ============================================================

mart_priorizacao = pd.read_csv(
    ARQUIVO_MART_PRIORIZACAO,
    dtype={
        "codigo_ibge": "string"
    }
)


print(
    "\nDimensão:",
    mart_priorizacao.shape
)


print(
    "Municípios:",
    mart_priorizacao[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "Duplicatas codigo_ibge:",
    mart_priorizacao
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)

ARQUIVO — MART PRIORIZAÇÃO

Arquivo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\priorizacao_agroambiental\base_modelo_priorizacao_agroambiental_soja_centro_oeste_sul_municipio.csv

Existe: True

Dimensão: (1505, 93)
Municípios: 1505
Duplicatas codigo_ibge: 0


In [17]:
# ============================================================
# VALIDAÇÃO ANALÍTICA — MART PRIORIZAÇÃO
# ============================================================

print("=" * 70)
print("VALIDAÇÃO — MART PRIORIZAÇÃO MUNICIPAL")
print("=" * 70)


print(
    "\nNULL totais:",
    mart_priorizacao
    .isna()
    .sum()
    .sum()
)


print(
    "\nElegibilidade:"
)


display(
    mart_priorizacao[
        "elegivel_cruzamento_priorizacao"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nQuadrantes:"
)


display(
    mart_priorizacao[
        "quadrante_priorizacao"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nConfiança:"
)


display(
    mart_priorizacao[
        "faixa_confianca_modelo"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nRobustez:"
)


display(
    mart_priorizacao[
        "faixa_robustez_priorizacao"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nPrioridade estratégica:"
)

print(
    mart_priorizacao[
        "flag_prioridade_estrategica_base"
    ]
    .sum()
)


print(
    "\nPrioridade robusta em 3 ou 4 cenários:"
)

print(
    mart_priorizacao[
        "flag_prioridade_robusta_3_ou_4_cenarios"
    ]
    .sum()
)

VALIDAÇÃO — MART PRIORIZAÇÃO MUNICIPAL

NULL totais: 3466

Elegibilidade:


elegivel_cruzamento_priorizacao
True     1400
False     105
Name: count, dtype: int64


Quadrantes:


quadrante_priorizacao
Alta relevância + Baixa pressão     475
Baixa relevância + Alta pressão     475
Alta relevância + Alta pressão      225
Baixa relevância + Baixa pressão    225
Dados insuficientes                 105
Name: count, dtype: int64


Confiança:


faixa_confianca_modelo
Alta                                   781
Moderada                               532
Não aplicável — dados insuficientes    105
Limitada                                87
Name: count, dtype: int64


Robustez:


faixa_robustez_priorizacao
Fora do estratégico — 0/4 cenários     1067
Muito robusta — 4/4 cenários            171
Não aplicável — dados insuficientes     105
Baixa robustez — 1/4 cenários            92
Robusta — 3/4 cenários                   48
Sensível — 2/4 cenários                  22
Name: count, dtype: int64


Prioridade estratégica:
225

Prioridade robusta em 3 ou 4 cenários:
219


In [18]:
# ============================================================
# INVENTÁRIO DE TIPOS — MART MUNICIPAL
# ============================================================

resumo_tipos_mart = (
    mart_priorizacao
    .dtypes
    .astype(str)
    .value_counts()
    .rename_axis(
        "dtype"
    )
    .reset_index(
        name="quantidade_colunas"
    )
)


print("=" * 70)
print("TIPOS — MART PRIORIZAÇÃO")
print("=" * 70)


display(
    resumo_tipos_mart
)


for dtype in mart_priorizacao.dtypes.unique():

    colunas_tipo = (
        mart_priorizacao
        .columns[
            mart_priorizacao.dtypes
            ==
            dtype
        ]
        .tolist()
    )


    print(
        "\n" + "-" * 70
    )

    print(
        "TIPO:",
        dtype
    )

    print(
        "QUANTIDADE:",
        len(
            colunas_tipo
        )
    )

    print(
        "-" * 70
    )


    for coluna in colunas_tipo:

        print(
            "->",
            coluna
        )

TIPOS — MART PRIORIZAÇÃO


,dtype,quantidade_colunas
0,float64,55
1,int64,14
2,object,12
3,bool,11
4,string,1



----------------------------------------------------------------------
TIPO: string
QUANTIDADE: 1
----------------------------------------------------------------------
-> codigo_ibge

----------------------------------------------------------------------
TIPO: object
QUANTIDADE: 12
----------------------------------------------------------------------
-> municipio
-> uf
-> regiao
-> quadrante_priorizacao
-> flag_ic95_taxa_cruza_zero
-> pior_qualidade_climatica
-> faixa_confianca_modelo
-> estrategico__base
-> estrategico__produtivo
-> estrategico__ambiental
-> estrategico__climatico
-> faixa_robustez_priorizacao

----------------------------------------------------------------------
TIPO: int64
QUANTIDADE: 14
----------------------------------------------------------------------
-> quantidade_anos
-> primeiro_ano_disponivel
-> ultimo_ano_disponivel
-> producao_soja_n_obs
-> rendimento_soja_n_obs
-> carbono_solo_n_obs
-> cobertura_natural_n_obs
-> seeg_co2e_completo_n_obs
-> periodo_i

In [19]:
# ============================================================
# AUDITORIA — COLUNAS BOOLEANAS NULLABLE
# ============================================================

COLUNAS_BOOLEANAS_NULLABLE_MART = [
    "flag_ic95_taxa_cruza_zero",
    "estrategico__base",
    "estrategico__produtivo",
    "estrategico__ambiental",
    "estrategico__climatico"
]


print("=" * 70)
print("BOOLEANOS NULLABLE — MART")
print("=" * 70)


for coluna in COLUNAS_BOOLEANAS_NULLABLE_MART:

    print(
        "\n" + "-" * 70
    )

    print(
        coluna
    )

    print(
        "-" * 70
    )


    print(
        "dtype atual:",
        mart_priorizacao[
            coluna
        ].dtype
    )


    print(
        "NULL:",
        mart_priorizacao[
            coluna
        ]
        .isna()
        .sum()
    )


    print(
        "Tipos Python encontrados:"
    )

    print(
        mart_priorizacao[
            coluna
        ]
        .dropna()
        .map(
            lambda valor:
                type(valor).__name__
        )
        .value_counts()
        .to_dict()
    )


    print(
        "Valores:"
    )


    display(
        mart_priorizacao[
            coluna
        ]
        .value_counts(
            dropna=False
        )
    )

BOOLEANOS NULLABLE — MART

----------------------------------------------------------------------
flag_ic95_taxa_cruza_zero
----------------------------------------------------------------------
dtype atual: object
NULL: 31
Tipos Python encontrados:
{'bool': 1474}
Valores:


flag_ic95_taxa_cruza_zero
False    1203
True      271
NaN        31
Name: count, dtype: int64


----------------------------------------------------------------------
estrategico__base
----------------------------------------------------------------------
dtype atual: object
NULL: 105
Tipos Python encontrados:
{'bool': 1400}
Valores:


estrategico__base
False    1175
True      225
NaN       105
Name: count, dtype: int64


----------------------------------------------------------------------
estrategico__produtivo
----------------------------------------------------------------------
dtype atual: object
NULL: 105
Tipos Python encontrados:
{'bool': 1400}
Valores:


estrategico__produtivo
False    1150
True      250
NaN       105
Name: count, dtype: int64


----------------------------------------------------------------------
estrategico__ambiental
----------------------------------------------------------------------
dtype atual: object
NULL: 105
Tipos Python encontrados:
{'bool': 1400}
Valores:


estrategico__ambiental
False    1158
True      242
NaN       105
Name: count, dtype: int64


----------------------------------------------------------------------
estrategico__climatico
----------------------------------------------------------------------
dtype atual: object
NULL: 105
Tipos Python encontrados:
{'bool': 1400}
Valores:


estrategico__climatico
False    1153
True      247
NaN       105
Name: count, dtype: int64

In [20]:
# ============================================================
# AUDITORIA DAS COLUNAS TEXTUAIS — MART
# ============================================================

COLUNAS_TEXTO_MART = [
    "municipio",
    "uf",
    "regiao",
    "quadrante_priorizacao",
    "pior_qualidade_climatica",
    "faixa_confianca_modelo",
    "faixa_robustez_priorizacao"
]


auditoria_textos_mart = []


for coluna in COLUNAS_TEXTO_MART:

    serie = (
        mart_priorizacao[
            coluna
        ]
        .dropna()
        .astype(str)
    )


    auditoria_textos_mart.append(
        {
            "coluna":
                coluna,

            "n_nao_null":
                len(
                    serie
                ),

            "n_distintos":
                serie.nunique(),

            "comprimento_maximo":
                serie
                .str.len()
                .max(),

            "exemplo":
                serie.iloc[0]
        }
    )


auditoria_textos_mart = (
    pd.DataFrame(
        auditoria_textos_mart
    )
    .sort_values(
        by="comprimento_maximo",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("=" * 70)
print("TEXTOS — MART PRIORIZAÇÃO")
print("=" * 70)


display(
    auditoria_textos_mart
)

TEXTOS — MART PRIORIZAÇÃO


,coluna,n_nao_null,n_distintos,comprimento_maximo,exemplo
0,faixa_confianca_modelo,1505,4,35,Moderada
1,faixa_robustez_priorizacao,1505,6,35,Fora do estratégico — 0/4 cenários
2,municipio,1505,1482,32,Abatiá
3,quadrante_priorizacao,1505,5,32,Alta relevância + Baixa pressão
4,regiao,1505,2,12,Sul
5,pior_qualidade_climatica,1505,5,11,baixa
6,uf,1505,7,2,PR


In [21]:
# ============================================================
# AUDITORIA — QUANTIDADE DE CENÁRIOS ESTRATÉGICOS
# ============================================================

serie_cenarios = (
    mart_priorizacao[
        "quantidade_cenarios_estrategico"
    ]
)


print("=" * 70)
print("QUANTIDADE DE CENÁRIOS ESTRATÉGICOS")
print("=" * 70)


print(
    "\ndtype atual:",
    serie_cenarios.dtype
)


print(
    "NULL:",
    serie_cenarios
    .isna()
    .sum()
)


print(
    "\nValores distintos:"
)


display(
    serie_cenarios
    .value_counts(
        dropna=False
    )
    .sort_index()
)


# ------------------------------------------------------------
# Conferir se todos os valores válidos são inteiros
# ------------------------------------------------------------

serie_cenarios_valida = (
    serie_cenarios
    .dropna()
)


valores_nao_inteiros = (
    (
        serie_cenarios_valida
        -
        serie_cenarios_valida.round()
    )
    .abs()
    >
    0.0000001
).sum()


print(
    "\nValores válidos não inteiros:",
    valores_nao_inteiros
)


print(
    "Mínimo:",
    serie_cenarios_valida.min()
)


print(
    "Máximo:",
    serie_cenarios_valida.max()
)

QUANTIDADE DE CENÁRIOS ESTRATÉGICOS

dtype atual: float64
NULL: 105

Valores distintos:


quantidade_cenarios_estrategico
0.0    1067
1.0      92
2.0      22
3.0      48
4.0     171
NaN     105
Name: count, dtype: int64


Valores válidos não inteiros: 0
Mínimo: 0.0
Máximo: 4.0


In [22]:
# ============================================================
# PREPARAR MART PARA CARGA MYSQL
# ============================================================

mart_priorizacao_mysql = (
    mart_priorizacao
    .copy()
)


# ------------------------------------------------------------
# Booleanos nullable:
# True / False / NULL
# ------------------------------------------------------------

for coluna in COLUNAS_BOOLEANAS_NULLABLE_MART:

    mart_priorizacao_mysql[
        coluna
    ] = (
        mart_priorizacao_mysql[
            coluna
        ]
        .astype(
            "boolean"
        )
    )


# ------------------------------------------------------------
# Quantidade de cenários:
# inteiro nullable 0–4 / NULL
# ------------------------------------------------------------

mart_priorizacao_mysql[
    "quantidade_cenarios_estrategico"
] = (
    mart_priorizacao_mysql[
        "quantidade_cenarios_estrategico"
    ]
    .astype(
        "Int8"
    )
)


print("=" * 70)
print("MART PREPARADO PARA MYSQL")
print("=" * 70)


print(
    "\nDimensão:",
    mart_priorizacao_mysql.shape
)


print(
    "\nTipos corrigidos:"
)


for coluna in [
    *COLUNAS_BOOLEANAS_NULLABLE_MART,
    "quantidade_cenarios_estrategico"
]:

    print(
        "->",
        coluna,
        ":",
        mart_priorizacao_mysql[
            coluna
        ].dtype
    )


print(
    "\nNULL totais antes:",
    mart_priorizacao
    .isna()
    .sum()
    .sum()
)


print(
    "NULL totais depois:",
    mart_priorizacao_mysql
    .isna()
    .sum()
    .sum()
)

MART PREPARADO PARA MYSQL

Dimensão: (1505, 93)

Tipos corrigidos:
-> flag_ic95_taxa_cruza_zero : boolean
-> estrategico__base : boolean
-> estrategico__produtivo : boolean
-> estrategico__ambiental : boolean
-> estrategico__climatico : boolean
-> quantidade_cenarios_estrategico : Int8

NULL totais antes: 3466
NULL totais depois: 3466


In [23]:
# ============================================================
# TIPOS SQL — MART PRIORIZAÇÃO MUNICIPAL
# ============================================================

from sqlalchemy.dialects.mysql import (
    CHAR,
    VARCHAR,
    SMALLINT,
    DOUBLE,
    BOOLEAN
)


TIPOS_SQL_MART = {}


# ------------------------------------------------------------
# Identificadores
# ------------------------------------------------------------

TIPOS_SQL_MART[
    "codigo_ibge"
] = CHAR(7)


TIPOS_SQL_MART[
    "uf"
] = CHAR(2)


# ------------------------------------------------------------
# Textos verdadeiros
# ------------------------------------------------------------

for coluna in COLUNAS_TEXTO_MART:

    if coluna == "uf":

        continue

    TIPOS_SQL_MART[
        coluna
    ] = VARCHAR(100)


# ------------------------------------------------------------
# Inteiros originais
# ------------------------------------------------------------

for coluna in mart_priorizacao_mysql.select_dtypes(
    include=[
        "int64"
    ]
).columns:

    TIPOS_SQL_MART[
        coluna
    ] = SMALLINT()


# ------------------------------------------------------------
# Inteiro nullable 0–4
# ------------------------------------------------------------

TIPOS_SQL_MART[
    "quantidade_cenarios_estrategico"
] = SMALLINT()


# ------------------------------------------------------------
# Decimais
# ------------------------------------------------------------

for coluna in mart_priorizacao_mysql.select_dtypes(
    include=[
        "float64"
    ]
).columns:

    TIPOS_SQL_MART[
        coluna
    ] = DOUBLE()


# ------------------------------------------------------------
# Booleanos normais e nullable
# ------------------------------------------------------------

for coluna in mart_priorizacao_mysql.select_dtypes(
    include=[
        "bool",
        "boolean"
    ]
).columns:

    TIPOS_SQL_MART[
        coluna
    ] = BOOLEAN()


# ------------------------------------------------------------
# Auditoria
# ------------------------------------------------------------

colunas_sem_tipo_mart = (
    set(
        mart_priorizacao_mysql.columns
    )
    -
    set(
        TIPOS_SQL_MART.keys()
    )
)


print("=" * 70)
print("MAPEAMENTO SQL — MART")
print("=" * 70)


print(
    "\nColunas da base:",
    len(
        mart_priorizacao_mysql.columns
    )
)


print(
    "Tipos SQL definidos:",
    len(
        TIPOS_SQL_MART
    )
)


print(
    "Colunas sem tipo:",
    len(
        colunas_sem_tipo_mart
    )
)


if colunas_sem_tipo_mart:

    print(
        sorted(
            colunas_sem_tipo_mart
        )
    )

MAPEAMENTO SQL — MART

Colunas da base: 93
Tipos SQL definidos: 93
Colunas sem tipo: 0


In [24]:
# ============================================================
# CARGA — MART PRIORIZAÇÃO MUNICIPAL
# ============================================================

NOME_TABELA_MART = (
    "mart_priorizacao_municipal"
)


inspector = inspect(
    engine
)


mart_ja_existe = (
    inspector.has_table(
        NOME_TABELA_MART
    )
)


print("=" * 70)
print("CARGA — MART PRIORIZAÇÃO MUNICIPAL")
print("=" * 70)


print(
    "\nTabela já existe:",
    mart_ja_existe
)


if mart_ja_existe:

    raise RuntimeError(
        "A tabela mart_priorizacao_municipal já existe. "
        "Carga interrompida para evitar sobrescrita."
    )


mart_priorizacao_mysql.to_sql(
    name=NOME_TABELA_MART,
    con=engine,
    if_exists="fail",
    index=False,
    dtype=TIPOS_SQL_MART,
    chunksize=200,
    method="multi"
)


print(
    "\nCarga concluída."
)

CARGA — MART PRIORIZAÇÃO MUNICIPAL

Tabela já existe: False

Carga concluída.


In [25]:
# ============================================================
# VALIDAÇÃO — MART DATAFRAME × MYSQL
# ============================================================

# ------------------------------------------------------------
# Expressão para contar TODOS os NULL da tabela
# ------------------------------------------------------------

expressoes_null_mart = []


for coluna in mart_priorizacao_mysql.columns:

    coluna_escapada = (
        coluna
        .replace(
            "`",
            "``"
        )
    )

    expressoes_null_mart.append(
        f"""
        SUM(
            CASE
                WHEN `{coluna_escapada}` IS NULL
                THEN 1
                ELSE 0
            END
        )
        """
    )


expressao_total_null_mart = (
    " + ".join(
        expressoes_null_mart
    )
)


with engine.connect() as conexao:

    # --------------------------------------------------------
    # Total de linhas
    # --------------------------------------------------------

    linhas_mart_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_MART}`;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Municípios únicos
    # --------------------------------------------------------

    municipios_mart_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(
                    DISTINCT codigo_ibge
                )
                FROM `{NOME_TABELA_MART}`;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Duplicatas codigo_ibge
    # --------------------------------------------------------

    duplicatas_mart_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM (
                    SELECT
                        codigo_ibge,
                        COUNT(*) AS quantidade
                    FROM `{NOME_TABELA_MART}`
                    GROUP BY codigo_ibge
                    HAVING COUNT(*) > 1
                ) AS duplicatas;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # NULL na futura PK
    # --------------------------------------------------------

    null_codigo_mart_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_MART}`
                WHERE codigo_ibge IS NULL;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Total de NULL
    # --------------------------------------------------------

    total_null_mart_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT
                    {expressao_total_null_mart}
                FROM `{NOME_TABELA_MART}`;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Controles analíticos principais
    # --------------------------------------------------------

    elegiveis_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_MART}`
                WHERE elegivel_cruzamento_priorizacao = TRUE;
                """
            )
        )
        .scalar()
    )


    estrategicos_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_MART}`
                WHERE flag_prioridade_estrategica_base = TRUE;
                """
            )
        )
        .scalar()
    )


    robustos_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_MART}`
                WHERE flag_prioridade_robusta_3_ou_4_cenarios = TRUE;
                """
            )
        )
        .scalar()
    )


# ============================================================
# COMPARAÇÃO
# ============================================================

total_null_mart_df = (
    mart_priorizacao_mysql
    .isna()
    .sum()
    .sum()
)


print("=" * 70)
print("VALIDAÇÃO MART — DATAFRAME × MYSQL")
print("=" * 70)


print(
    "\nLinhas DataFrame:",
    len(mart_priorizacao_mysql)
)

print(
    "Linhas MySQL:",
    linhas_mart_mysql
)


print(
    "\nMunicípios DataFrame:",
    mart_priorizacao_mysql[
        "codigo_ibge"
    ].nunique()
)

print(
    "Municípios MySQL:",
    municipios_mart_mysql
)


print(
    "\nDuplicatas codigo_ibge:",
    duplicatas_mart_mysql
)


print(
    "NULL codigo_ibge:",
    null_codigo_mart_mysql
)


print(
    "\nNULL totais DataFrame:",
    total_null_mart_df
)

print(
    "NULL totais MySQL:",
    total_null_mart_mysql
)


print(
    "NULL preservados corretamente:",
    total_null_mart_df
    ==
    total_null_mart_mysql
)


print(
    "\nElegíveis MySQL:",
    elegiveis_mysql
)


print(
    "Estratégicos MySQL:",
    estrategicos_mysql
)


print(
    "Robustos >= 3/4 MySQL:",
    robustos_mysql
)

VALIDAÇÃO MART — DATAFRAME × MYSQL

Linhas DataFrame: 1505
Linhas MySQL: 1505

Municípios DataFrame: 1505
Municípios MySQL: 1505

Duplicatas codigo_ibge: 0
NULL codigo_ibge: 0

NULL totais DataFrame: 3466
NULL totais MySQL: 3466
NULL preservados corretamente: True

Elegíveis MySQL: 1400
Estratégicos MySQL: 225
Robustos >= 3/4 MySQL: 219


In [26]:
# ============================================================
# VALIDAÇÃO — BOOLEANOS NULLABLE NO MYSQL
# ============================================================

auditoria_booleanos_mysql = []


for coluna in COLUNAS_BOOLEANAS_NULLABLE_MART:

    with engine.connect() as conexao:

        resultado = (
            conexao.execute(
                text(
                    f"""
                    SELECT
                        SUM(
                            CASE
                                WHEN `{coluna}` = TRUE
                                THEN 1
                                ELSE 0
                            END
                        ) AS qtd_true,

                        SUM(
                            CASE
                                WHEN `{coluna}` = FALSE
                                THEN 1
                                ELSE 0
                            END
                        ) AS qtd_false,

                        SUM(
                            CASE
                                WHEN `{coluna}` IS NULL
                                THEN 1
                                ELSE 0
                            END
                        ) AS qtd_null

                    FROM `{NOME_TABELA_MART}`;
                    """
                )
            )
            .one()
        )


    auditoria_booleanos_mysql.append(
        {
            "coluna":
                coluna,

            "true_mysql":
                int(
                    resultado[0]
                ),

            "false_mysql":
                int(
                    resultado[1]
                ),

            "null_mysql":
                int(
                    resultado[2]
                ),

            "true_dataframe":
                int(
                    (
                        mart_priorizacao_mysql[
                            coluna
                        ]
                        == True
                    )
                    .sum()
                ),

            "false_dataframe":
                int(
                    (
                        mart_priorizacao_mysql[
                            coluna
                        ]
                        == False
                    )
                    .sum()
                ),

            "null_dataframe":
                int(
                    mart_priorizacao_mysql[
                        coluna
                    ]
                    .isna()
                    .sum()
                )
        }
    )


auditoria_booleanos_mysql = (
    pd.DataFrame(
        auditoria_booleanos_mysql
    )
)


auditoria_booleanos_mysql[
    "confere"
] = (
    (
        auditoria_booleanos_mysql[
            "true_mysql"
        ]
        ==
        auditoria_booleanos_mysql[
            "true_dataframe"
        ]
    )
    &
    (
        auditoria_booleanos_mysql[
            "false_mysql"
        ]
        ==
        auditoria_booleanos_mysql[
            "false_dataframe"
        ]
    )
    &
    (
        auditoria_booleanos_mysql[
            "null_mysql"
        ]
        ==
        auditoria_booleanos_mysql[
            "null_dataframe"
        ]
    )
)


print("=" * 70)
print("BOOLEANOS NULLABLE — DATAFRAME × MYSQL")
print("=" * 70)


display(
    auditoria_booleanos_mysql
)


print(
    "\nTodas as colunas conferem:",
    auditoria_booleanos_mysql[
        "confere"
    ]
    .all()
)

BOOLEANOS NULLABLE — DATAFRAME × MYSQL


,coluna,true_mysql,false_mysql,null_mysql,true_dataframe,false_dataframe,null_dataframe,confere
0,flag_ic95_taxa_cruza_zero,271,1203,31,271,1203,31,True
1,estrategico__base,225,1175,105,225,1175,105,True
2,estrategico__produtivo,250,1150,105,250,1150,105,True
3,estrategico__ambiental,242,1158,105,242,1158,105,True
4,estrategico__climatico,247,1153,105,247,1153,105,True



Todas as colunas conferem: True


In [27]:
# ============================================================
# VALIDAÇÃO — QUANTIDADE DE CENÁRIOS NO MYSQL
# ============================================================

distribuicao_cenarios_mysql = pd.read_sql(
    text(
        f"""
        SELECT
            quantidade_cenarios_estrategico,
            COUNT(*) AS quantidade
        FROM `{NOME_TABELA_MART}`
        GROUP BY
            quantidade_cenarios_estrategico
        ORDER BY
            quantidade_cenarios_estrategico;
        """
    ),
    engine
)


print("=" * 70)
print("CENÁRIOS ESTRATÉGICOS — MYSQL")
print("=" * 70)


display(
    distribuicao_cenarios_mysql
)


with engine.connect() as conexao:

    null_cenarios_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_MART}`
                WHERE
                    quantidade_cenarios_estrategico
                    IS NULL;
                """
            )
        )
        .scalar()
    )


print(
    "\nNULL em quantidade_cenarios_estrategico:",
    null_cenarios_mysql
)

CENÁRIOS ESTRATÉGICOS — MYSQL


,quantidade_cenarios_estrategico,quantidade
0,NaN,105
1,0.0,1067
2,1.0,92
3,2.0,22
4,3.0,48
5,4.0,171



NULL em quantidade_cenarios_estrategico: 105


In [28]:
# ============================================================
# PRIMARY KEY E ÍNDICES — MART PRIORIZAÇÃO MUNICIPAL
# ============================================================

from sqlalchemy import inspect, text


inspector = inspect(
    engine
)


# ------------------------------------------------------------
# Verificar PK existente
# ------------------------------------------------------------

pk_mart_atual = (
    inspector
    .get_pk_constraint(
        NOME_TABELA_MART
    )
)


colunas_pk_mart = (
    pk_mart_atual
    .get(
        "constrained_columns"
    )
    or []
)


print("=" * 70)
print("PK E ÍNDICES — MART PRIORIZAÇÃO")
print("=" * 70)


print(
    "\nPK antes:",
    colunas_pk_mart
)


# ------------------------------------------------------------
# Criar PK
# ------------------------------------------------------------

if not colunas_pk_mart:

    with engine.begin() as conexao:

        conexao.execute(
            text(
                f"""
                ALTER TABLE `{NOME_TABELA_MART}`

                MODIFY `codigo_ibge`
                    CHAR(7) NOT NULL,

                ADD PRIMARY KEY (
                    `codigo_ibge`
                );
                """
            )
        )

    print(
        "Primary Key criada."
    )

else:

    print(
        "Primary Key já existe."
    )


# ------------------------------------------------------------
# Atualizar inspector
# ------------------------------------------------------------

inspector = inspect(
    engine
)


indices_existentes_mart = {
    indice["name"]
    for indice in inspector.get_indexes(
        NOME_TABELA_MART
    )
}


# ------------------------------------------------------------
# Índices desejados
# ------------------------------------------------------------

indices_mart = {

    "idx_mart_uf":
        ["uf"],

    "idx_mart_regiao":
        ["regiao"],

    "idx_mart_quadrante":
        ["quadrante_priorizacao"],

    "idx_mart_confianca":
        ["faixa_confianca_modelo"],

    "idx_mart_robustez":
        ["faixa_robustez_priorizacao"]
}


for nome_indice, colunas in indices_mart.items():

    if nome_indice not in indices_existentes_mart:

        colunas_sql = ", ".join(
            f"`{coluna}`"
            for coluna in colunas
        )

        with engine.begin() as conexao:

            conexao.execute(
                text(
                    f"""
                    CREATE INDEX
                        `{nome_indice}`

                    ON `{NOME_TABELA_MART}`
                        ({colunas_sql});
                    """
                )
            )

        print(
            "Índice criado:",
            nome_indice
        )


print(
    "\nEstrutura concluída."
)

PK E ÍNDICES — MART PRIORIZAÇÃO

PK antes: []
Primary Key criada.
Índice criado: idx_mart_uf
Índice criado: idx_mart_regiao
Índice criado: idx_mart_quadrante
Índice criado: idx_mart_confianca
Índice criado: idx_mart_robustez

Estrutura concluída.


In [29]:
# ============================================================
# AUDITORIA FINAL — MART PRIORIZAÇÃO MUNICIPAL
# ============================================================

inspector = inspect(
    engine
)


pk_mart_final = (
    inspector
    .get_pk_constraint(
        NOME_TABELA_MART
    )
)


indices_mart_final = (
    inspector
    .get_indexes(
        NOME_TABELA_MART
    )
)


print("=" * 70)
print("ESTRUTURA FINAL — MART PRIORIZAÇÃO")
print("=" * 70)


print(
    "\nPrimary Key:"
)

print(
    pk_mart_final
)


print(
    "\nÍndices:"
)


for indice in indices_mart_final:

    print(
        "->",
        indice["name"],
        "|",
        indice["column_names"]
    )


# ------------------------------------------------------------
# Estrutura da chave
# ------------------------------------------------------------

estrutura_codigo_mart = pd.read_sql(
    text(
        """
        SELECT
            COLUMN_NAME,
            COLUMN_TYPE,
            IS_NULLABLE,
            COLUMN_KEY
        FROM information_schema.COLUMNS
        WHERE
            TABLE_SCHEMA = 'agroesg_analytics'
            AND TABLE_NAME = 'mart_priorizacao_municipal'
            AND COLUMN_NAME = 'codigo_ibge';
        """
    ),
    engine
)


print(
    "\nEstrutura da chave:"
)


display(
    estrutura_codigo_mart
)


# ------------------------------------------------------------
# Amostra estratégica
# ------------------------------------------------------------

amostra_mart_mysql = pd.read_sql(
    text(
        """
        SELECT
            codigo_ibge,
            municipio,
            uf,
            regiao,
            score_relevancia_produtiva,
            score_pressao_agroambiental,
            quadrante_priorizacao,
            faixa_confianca_modelo,
            quantidade_cenarios_estrategico,
            faixa_robustez_priorizacao,
            flag_prioridade_estrategica_base
        FROM mart_priorizacao_municipal
        WHERE
            flag_prioridade_estrategica_base = TRUE
        ORDER BY
            score_relevancia_produtiva DESC
        LIMIT 10;
        """
    ),
    engine
)


print(
    "\nAmostra de municípios estratégicos:"
)


display(
    amostra_mart_mysql
)

ESTRUTURA FINAL — MART PRIORIZAÇÃO

Primary Key:
{'constrained_columns': ['codigo_ibge'], 'name': None}

Índices:
-> idx_mart_confianca | ['faixa_confianca_modelo']
-> idx_mart_quadrante | ['quadrante_priorizacao']
-> idx_mart_regiao | ['regiao']
-> idx_mart_robustez | ['faixa_robustez_priorizacao']
-> idx_mart_uf | ['uf']

Estrutura da chave:


,COLUMN_NAME,COLUMN_TYPE,IS_NULLABLE,COLUMN_KEY
0,codigo_ibge,char(7),NO,PRI



Amostra de municípios estratégicos:


,codigo_ibge,municipio,uf,regiao,score_relevancia_produtiva,score_pressao_agroambiental,quadrante_priorizacao,faixa_confianca_modelo,quantidade_cenarios_estrategico,faixa_robustez_priorizacao,flag_prioridade_estrategica_base
0,5108501,Vera,MT,Centro-Oeste,88.170122,63.150256,Alta relevância + Alta pressão,Limitada,4,Muito robusta — 4/4 cenários,1
1,4312617,Muitos Capões,RS,Sul,87.800810,52.931927,Alta relevância + Alta pressão,Alta,4,Muito robusta — 4/4 cenários,1
2,5107909,Sinop,MT,Centro-Oeste,86.907315,60.118109,Alta relevância + Alta pressão,Limitada,4,Muito robusta — 4/4 cenários,1
3,5106240,Nova Ubiratã,MT,Centro-Oeste,84.846319,62.807670,Alta relevância + Alta pressão,Limitada,4,Muito robusta — 4/4 cenários,1
4,5101852,Bom Jesus do Araguaia,MT,Centro-Oeste,84.798666,57.432346,Alta relevância + Alta pressão,Limitada,4,Muito robusta — 4/4 cenários,1
5,5104526,Ipiranga do Norte,MT,Centro-Oeste,83.321420,54.362546,Alta relevância + Alta pressão,Limitada,4,Muito robusta — 4/4 cenários,1
6,5103056,Cláudia,MT,Centro-Oeste,81.736955,67.532028,Alta relevância + Alta pressão,Limitada,4,Muito robusta — 4/4 cenários,1
7,5212501,Luziânia,GO,Centro-Oeste,81.224684,49.996092,Alta relevância + Alta pressão,Alta,4,Muito robusta — 4/4 cenários,1
8,5106224,Nova Mutum,MT,Centro-Oeste,81.153205,52.423634,Alta relevância + Alta pressão,Moderada,4,Muito robusta — 4/4 cenários,1
9,5107859,São Félix do Araguaia,MT,Centro-Oeste,81.010245,59.413833,Alta relevância + Alta pressão,Limitada,4,Muito robusta — 4/4 cenários,1


In [30]:
# ============================================================
# INVENTÁRIO ATUAL — MYSQL
# ============================================================

tabelas_mysql = pd.read_sql(
    text(
        """
        SELECT
            TABLE_NAME,
            TABLE_ROWS
        FROM information_schema.TABLES
        WHERE
            TABLE_SCHEMA = 'agroesg_analytics'
        ORDER BY
            TABLE_NAME;
        """
    ),
    engine
)


print("=" * 70)
print("TABELAS ATUAIS — AGROESG ANALYTICS")
print("=" * 70)


display(
    tabelas_mysql
)

TABELAS ATUAIS — AGROESG ANALYTICS


,TABLE_NAME,TABLE_ROWS
0,fato_agroambiental_anual,7953
1,mart_priorizacao_municipal,1453


In [31]:
# ============================================================
# INVENTÁRIO EXATO — TABELAS PRINCIPAIS MYSQL
# ============================================================

inventario_exato_mysql = pd.read_sql(
    text(
        """
        SELECT
            'fato_agroambiental_anual' AS tabela,
            COUNT(*) AS quantidade_registros
        FROM fato_agroambiental_anual

        UNION ALL

        SELECT
            'mart_priorizacao_municipal' AS tabela,
            COUNT(*) AS quantidade_registros
        FROM mart_priorizacao_municipal;
        """
    ),
    engine
)


print("=" * 70)
print("INVENTÁRIO EXATO — MYSQL")
print("=" * 70)


display(
    inventario_exato_mysql
)

INVENTÁRIO EXATO — MYSQL


,tabela,quantidade_registros
0,fato_agroambiental_anual,8674
1,mart_priorizacao_municipal,1505


In [32]:
# ============================================================
# CARREGAR — TESTE DE SENSIBILIDADE
# ============================================================

ARQUIVO_SENSIBILIDADE = (
    RAIZ_PROJETO
    /
    "data"
    /
    "databases_curated"
    /
    "priorizacao_agroambiental"
    /
    "teste_sensibilidade_priorizacao_soja_centro_oeste_sul.csv"
)


print("=" * 70)
print("ARQUIVO — TESTE DE SENSIBILIDADE")
print("=" * 70)


print(
    "\nArquivo:"
)

print(
    ARQUIVO_SENSIBILIDADE
)


print(
    "\nExiste:",
    ARQUIVO_SENSIBILIDADE.exists()
)


sensibilidade = pd.read_csv(
    ARQUIVO_SENSIBILIDADE,
    dtype={
        "codigo_ibge": "string"
    }
)


print(
    "\nDimensão:",
    sensibilidade.shape
)


print(
    "Municípios:",
    sensibilidade[
        "codigo_ibge"
    ].nunique()
)


print(
    "Duplicatas codigo_ibge:",
    sensibilidade
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "NULL totais:",
    sensibilidade
    .isna()
    .sum()
    .sum()
)

ARQUIVO — TESTE DE SENSIBILIDADE

Arquivo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\priorizacao_agroambiental\teste_sensibilidade_priorizacao_soja_centro_oeste_sul.csv

Existe: True

Dimensão: (1400, 16)
Municípios: 1400
Duplicatas codigo_ibge: 0
NULL totais: 0


In [33]:
# ============================================================
# INVENTÁRIO — TESTE DE SENSIBILIDADE
# ============================================================

print("=" * 70)
print("ESTRUTURA — TESTE DE SENSIBILIDADE")
print("=" * 70)


resumo_tipos_sensibilidade = (
    sensibilidade
    .dtypes
    .astype(str)
    .value_counts()
    .rename_axis(
        "dtype"
    )
    .reset_index(
        name="quantidade_colunas"
    )
)


display(
    resumo_tipos_sensibilidade
)


print(
    "\nColunas:"
)


for coluna in sensibilidade.columns:

    print(
        "->",
        coluna,
        "|",
        sensibilidade[
            coluna
        ].dtype
    )

ESTRUTURA — TESTE DE SENSIBILIDADE


,dtype,quantidade_colunas
0,float64,8
1,bool,4
2,object,3
3,string,1



Colunas:
-> codigo_ibge | string
-> municipio | object
-> uf | object
-> regiao | object
-> score_relevancia__base | float64
-> score_pressao__base | float64
-> estrategico__base | bool
-> score_relevancia__produtivo | float64
-> score_pressao__produtivo | float64
-> estrategico__produtivo | bool
-> score_relevancia__ambiental | float64
-> score_pressao__ambiental | float64
-> estrategico__ambiental | bool
-> score_relevancia__climatico | float64
-> score_pressao__climatico | float64
-> estrategico__climatico | bool


In [34]:
# ============================================================
# TIPOS SQL — FATO SENSIBILIDADE PRIORIZAÇÃO
# ============================================================

from sqlalchemy.dialects.mysql import (
    CHAR,
    VARCHAR,
    DOUBLE,
    BOOLEAN
)


TIPOS_SQL_SENSIBILIDADE = {}


# ------------------------------------------------------------
# Identificadores e textos
# ------------------------------------------------------------

TIPOS_SQL_SENSIBILIDADE[
    "codigo_ibge"
] = CHAR(7)


TIPOS_SQL_SENSIBILIDADE[
    "municipio"
] = VARCHAR(100)


TIPOS_SQL_SENSIBILIDADE[
    "uf"
] = CHAR(2)


TIPOS_SQL_SENSIBILIDADE[
    "regiao"
] = VARCHAR(32)


# ------------------------------------------------------------
# Scores
# ------------------------------------------------------------

for coluna in sensibilidade.select_dtypes(
    include=[
        "float64"
    ]
).columns:

    TIPOS_SQL_SENSIBILIDADE[
        coluna
    ] = DOUBLE()


# ------------------------------------------------------------
# Flags booleanas
# ------------------------------------------------------------

for coluna in sensibilidade.select_dtypes(
    include=[
        "bool"
    ]
).columns:

    TIPOS_SQL_SENSIBILIDADE[
        coluna
    ] = BOOLEAN()


# ------------------------------------------------------------
# Auditoria
# ------------------------------------------------------------

colunas_sem_tipo_sensibilidade = (
    set(
        sensibilidade.columns
    )
    -
    set(
        TIPOS_SQL_SENSIBILIDADE.keys()
    )
)


print("=" * 70)
print("MAPEAMENTO SQL — SENSIBILIDADE")
print("=" * 70)


print(
    "\nColunas da base:",
    len(
        sensibilidade.columns
    )
)


print(
    "Tipos SQL definidos:",
    len(
        TIPOS_SQL_SENSIBILIDADE
    )
)


print(
    "Colunas sem tipo:",
    len(
        colunas_sem_tipo_sensibilidade
    )
)


if colunas_sem_tipo_sensibilidade:

    print(
        sorted(
            colunas_sem_tipo_sensibilidade
        )
    )

MAPEAMENTO SQL — SENSIBILIDADE

Colunas da base: 16
Tipos SQL definidos: 16
Colunas sem tipo: 0


In [35]:
# ============================================================
# CARGA — FATO SENSIBILIDADE PRIORIZAÇÃO
# ============================================================

NOME_TABELA_SENSIBILIDADE = (
    "fato_sensibilidade_priorizacao"
)


inspector = inspect(
    engine
)


sensibilidade_ja_existe = (
    inspector.has_table(
        NOME_TABELA_SENSIBILIDADE
    )
)


print("=" * 70)
print("CARGA — FATO SENSIBILIDADE")
print("=" * 70)


print(
    "\nTabela já existe:",
    sensibilidade_ja_existe
)


if sensibilidade_ja_existe:

    raise RuntimeError(
        "A tabela fato_sensibilidade_priorizacao já existe. "
        "Carga interrompida para evitar sobrescrita."
    )


sensibilidade.to_sql(
    name=NOME_TABELA_SENSIBILIDADE,
    con=engine,
    if_exists="fail",
    index=False,
    dtype=TIPOS_SQL_SENSIBILIDADE,
    chunksize=200,
    method="multi"
)


print(
    "\nCarga concluída."
)

CARGA — FATO SENSIBILIDADE

Tabela já existe: False

Carga concluída.


In [36]:
# ============================================================
# VALIDAÇÃO — SENSIBILIDADE DATAFRAME × MYSQL
# ============================================================

with engine.connect() as conexao:

    # --------------------------------------------------------
    # Linhas
    # --------------------------------------------------------

    linhas_sens_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_SENSIBILIDADE}`;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Municípios distintos
    # --------------------------------------------------------

    municipios_sens_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(
                    DISTINCT codigo_ibge
                )
                FROM `{NOME_TABELA_SENSIBILIDADE}`;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Duplicatas
    # --------------------------------------------------------

    duplicatas_sens_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM (
                    SELECT
                        codigo_ibge,
                        COUNT(*) AS quantidade
                    FROM `{NOME_TABELA_SENSIBILIDADE}`
                    GROUP BY codigo_ibge
                    HAVING COUNT(*) > 1
                ) AS duplicatas;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # NULL totais
    # --------------------------------------------------------

    expressoes_null = []

    for coluna in sensibilidade.columns:

        expressoes_null.append(
            f"""
            SUM(
                CASE
                    WHEN `{coluna}` IS NULL
                    THEN 1
                    ELSE 0
                END
            )
            """
        )


    expressao_null_total = (
        " + ".join(
            expressoes_null
        )
    )


    total_null_sens_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT
                    {expressao_null_total}
                FROM `{NOME_TABELA_SENSIBILIDADE}`;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Quantidade estratégica em cada cenário
    # --------------------------------------------------------

    contagens_cenarios = (
        conexao.execute(
            text(
                f"""
                SELECT
                    SUM(
                        CASE
                            WHEN estrategico__base = TRUE
                            THEN 1 ELSE 0
                        END
                    ) AS base,

                    SUM(
                        CASE
                            WHEN estrategico__produtivo = TRUE
                            THEN 1 ELSE 0
                        END
                    ) AS produtivo,

                    SUM(
                        CASE
                            WHEN estrategico__ambiental = TRUE
                            THEN 1 ELSE 0
                        END
                    ) AS ambiental,

                    SUM(
                        CASE
                            WHEN estrategico__climatico = TRUE
                            THEN 1 ELSE 0
                        END
                    ) AS climatico

                FROM `{NOME_TABELA_SENSIBILIDADE}`;
                """
            )
        )
        .mappings()
        .one()
    )


    # --------------------------------------------------------
    # Códigos que não existem no Mart
    # --------------------------------------------------------

    codigos_orfaos = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_SENSIBILIDADE}` s
                LEFT JOIN mart_priorizacao_municipal m
                    ON s.codigo_ibge = m.codigo_ibge
                WHERE m.codigo_ibge IS NULL;
                """
            )
        )
        .scalar()
    )


    # --------------------------------------------------------
    # Municípios da sensibilidade que não estão elegíveis
    # --------------------------------------------------------

    nao_elegiveis_no_mart = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_SENSIBILIDADE}` s
                INNER JOIN mart_priorizacao_municipal m
                    ON s.codigo_ibge = m.codigo_ibge
                WHERE
                    m.elegivel_cruzamento_priorizacao = FALSE;
                """
            )
        )
        .scalar()
    )


print("=" * 70)
print("VALIDAÇÃO — FATO SENSIBILIDADE")
print("=" * 70)


print(
    "\nLinhas DataFrame:",
    len(
        sensibilidade
    )
)

print(
    "Linhas MySQL:",
    linhas_sens_mysql
)


print(
    "\nMunicípios DataFrame:",
    sensibilidade[
        "codigo_ibge"
    ].nunique()
)

print(
    "Municípios MySQL:",
    municipios_sens_mysql
)


print(
    "\nDuplicatas codigo_ibge:",
    duplicatas_sens_mysql
)


print(
    "NULL totais MySQL:",
    total_null_sens_mysql
)


print(
    "\nEstratégicos — cenário base:",
    contagens_cenarios[
        "base"
    ]
)

print(
    "Estratégicos — produtivo:",
    contagens_cenarios[
        "produtivo"
    ]
)

print(
    "Estratégicos — ambiental:",
    contagens_cenarios[
        "ambiental"
    ]
)

print(
    "Estratégicos — climático:",
    contagens_cenarios[
        "climatico"
    ]
)


print(
    "\nCódigos inexistentes no Mart:",
    codigos_orfaos
)


print(
    "Registros não elegíveis no Mart:",
    nao_elegiveis_no_mart
)

VALIDAÇÃO — FATO SENSIBILIDADE

Linhas DataFrame: 1400
Linhas MySQL: 1400

Municípios DataFrame: 1400
Municípios MySQL: 1400

Duplicatas codigo_ibge: 0
NULL totais MySQL: 0

Estratégicos — cenário base: 225
Estratégicos — produtivo: 250
Estratégicos — ambiental: 242
Estratégicos — climático: 247

Códigos inexistentes no Mart: 0
Registros não elegíveis no Mart: 0


In [37]:
# ============================================================
# PK + FK — FATO SENSIBILIDADE PRIORIZAÇÃO
# ============================================================

from sqlalchemy import inspect, text


inspector = inspect(
    engine
)


# ------------------------------------------------------------
# PRIMARY KEY atual
# ------------------------------------------------------------

pk_sens_atual = (
    inspector
    .get_pk_constraint(
        NOME_TABELA_SENSIBILIDADE
    )
)


colunas_pk_sens = (
    pk_sens_atual
    .get(
        "constrained_columns"
    )
    or []
)


print("=" * 70)
print("PK + FK — FATO SENSIBILIDADE")
print("=" * 70)


print(
    "\nPK antes:",
    colunas_pk_sens
)


# ------------------------------------------------------------
# Criar PRIMARY KEY
# ------------------------------------------------------------

if not colunas_pk_sens:

    with engine.begin() as conexao:

        conexao.execute(
            text(
                f"""
                ALTER TABLE `{NOME_TABELA_SENSIBILIDADE}`

                MODIFY `codigo_ibge`
                    CHAR(7) NOT NULL,

                ADD PRIMARY KEY (
                    `codigo_ibge`
                );
                """
            )
        )

    print(
        "Primary Key criada."
    )

else:

    print(
        "Primary Key já existe."
    )


# ------------------------------------------------------------
# Atualizar inspector
# ------------------------------------------------------------

inspector = inspect(
    engine
)


fks_existentes = (
    inspector
    .get_foreign_keys(
        NOME_TABELA_SENSIBILIDADE
    )
)


nomes_fks_existentes = {
    fk["name"]
    for fk in fks_existentes
    if fk["name"] is not None
}


NOME_FK_SENS_MART = (
    "fk_sensibilidade_mart_codigo_ibge"
)


# ------------------------------------------------------------
# Criar FOREIGN KEY
# ------------------------------------------------------------

if NOME_FK_SENS_MART not in nomes_fks_existentes:

    with engine.begin() as conexao:

        conexao.execute(
            text(
                f"""
                ALTER TABLE `{NOME_TABELA_SENSIBILIDADE}`

                ADD CONSTRAINT
                    `{NOME_FK_SENS_MART}`

                FOREIGN KEY (
                    `codigo_ibge`
                )

                REFERENCES
                    `mart_priorizacao_municipal`
                    (`codigo_ibge`);
                """
            )
        )

    print(
        "Foreign Key criada."
    )

else:

    print(
        "Foreign Key já existe."
    )


print(
    "\nEstrutura concluída."
)

PK + FK — FATO SENSIBILIDADE

PK antes: []
Primary Key criada.
Foreign Key criada.

Estrutura concluída.


In [38]:
# ============================================================
# AUDITORIA FINAL — FATO SENSIBILIDADE
# ============================================================

inspector = inspect(
    engine
)


pk_sens_final = (
    inspector
    .get_pk_constraint(
        NOME_TABELA_SENSIBILIDADE
    )
)


fk_sens_final = (
    inspector
    .get_foreign_keys(
        NOME_TABELA_SENSIBILIDADE
    )
)


print("=" * 70)
print("ESTRUTURA FINAL — FATO SENSIBILIDADE")
print("=" * 70)


print(
    "\nPrimary Key:"
)

print(
    pk_sens_final
)


print(
    "\nForeign Keys:"
)


for fk in fk_sens_final:

    print(
        "-> Nome:",
        fk["name"]
    )

    print(
        "   Coluna:",
        fk["constrained_columns"]
    )

    print(
        "   Referência:",
        fk["referred_table"],
        fk["referred_columns"]
    )


# ------------------------------------------------------------
# Contagem exata
# ------------------------------------------------------------

with engine.connect() as conexao:

    total_sens_final = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_SENSIBILIDADE}`;
                """
            )
        )
        .scalar()
    )


print(
    "\nRegistros:",
    total_sens_final
)

ESTRUTURA FINAL — FATO SENSIBILIDADE

Primary Key:
{'constrained_columns': ['codigo_ibge'], 'name': None}

Foreign Keys:
-> Nome: fk_sensibilidade_mart_codigo_ibge
   Coluna: ['codigo_ibge']
   Referência: mart_priorizacao_municipal ['codigo_ibge']

Registros: 1400


In [39]:
# ============================================================
# CARREGAR E AUDITAR — RESUMO DE SENSIBILIDADE
# ============================================================

ARQUIVO_RESUMO_SENSIBILIDADE = (
    RAIZ_PROJETO
    /
    "data"
    /
    "databases_curated"
    /
    "priorizacao_agroambiental"
    /
    "resumo_sensibilidade_priorizacao_soja_centro_oeste_sul.csv"
)


print("=" * 70)
print("ARQUIVO — RESUMO DE SENSIBILIDADE")
print("=" * 70)


print(
    "\nArquivo:"
)

print(
    ARQUIVO_RESUMO_SENSIBILIDADE
)


print(
    "\nExiste:",
    ARQUIVO_RESUMO_SENSIBILIDADE.exists()
)


resumo_sensibilidade = pd.read_csv(
    ARQUIVO_RESUMO_SENSIBILIDADE
)


print(
    "\nDimensão:",
    resumo_sensibilidade.shape
)


print(
    "NULL totais:",
    resumo_sensibilidade
    .isna()
    .sum()
    .sum()
)


print(
    "\nColunas e tipos:"
)


for coluna in resumo_sensibilidade.columns:

    print(
        "->",
        coluna,
        "|",
        resumo_sensibilidade[
            coluna
        ].dtype
    )


print(
    "\nConteúdo:"
)


display(
    resumo_sensibilidade
)

ARQUIVO — RESUMO DE SENSIBILIDADE

Arquivo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\priorizacao_agroambiental\resumo_sensibilidade_priorizacao_soja_centro_oeste_sul.csv

Existe: True

Dimensão: (4, 7)
NULL totais: 0

Colunas e tipos:
-> cenario | object
-> estrategicos_cenario | int64
-> estrategicos_em_comum_com_base | int64
-> retencao_dos_225_base_pct | float64
-> jaccard_pct | float64
-> spearman_relevancia_vs_base | float64
-> spearman_pressao_vs_base | float64

Conteúdo:


,cenario,estrategicos_cenario,estrategicos_em_comum_com_base,retencao_dos_225_base_pct,jaccard_pct,spearman_relevancia_vs_base,spearman_pressao_vs_base
0,base,225,225,100.000000,100.000000,1.000000,1.000000
1,produtivo,250,216,96.000000,83.397683,0.985597,1.000000
2,ambiental,242,191,84.888889,69.202899,1.000000,0.917803
3,climatico,247,207,92.000000,78.113208,1.000000,0.949328


In [40]:
# ============================================================
# TIPOS SQL — DIM CENÁRIO SENSIBILIDADE
# ============================================================

from sqlalchemy.dialects.mysql import (
    VARCHAR,
    SMALLINT,
    DOUBLE
)


TIPOS_SQL_CENARIO = {
    "cenario":
        VARCHAR(20),

    "estrategicos_cenario":
        SMALLINT(),

    "estrategicos_em_comum_com_base":
        SMALLINT(),

    "retencao_dos_225_base_pct":
        DOUBLE(),

    "jaccard_pct":
        DOUBLE(),

    "spearman_relevancia_vs_base":
        DOUBLE(),

    "spearman_pressao_vs_base":
        DOUBLE()
}


# ------------------------------------------------------------
# Auditoria
# ------------------------------------------------------------

colunas_sem_tipo_cenario = (
    set(
        resumo_sensibilidade.columns
    )
    -
    set(
        TIPOS_SQL_CENARIO.keys()
    )
)


print("=" * 70)
print("MAPEAMENTO SQL — CENÁRIOS")
print("=" * 70)


print(
    "\nColunas:",
    len(
        resumo_sensibilidade.columns
    )
)


print(
    "Tipos definidos:",
    len(
        TIPOS_SQL_CENARIO
    )
)


print(
    "Colunas sem tipo:",
    len(
        colunas_sem_tipo_cenario
    )
)


print(
    "\nCenários únicos:",
    resumo_sensibilidade[
        "cenario"
    ].nunique()
)


print(
    "Duplicatas de cenário:",
    resumo_sensibilidade
    .duplicated(
        subset=[
            "cenario"
        ]
    )
    .sum()
)

MAPEAMENTO SQL — CENÁRIOS

Colunas: 7
Tipos definidos: 7
Colunas sem tipo: 0

Cenários únicos: 4
Duplicatas de cenário: 0


In [41]:
# ============================================================
# CARGA — DIM CENÁRIO SENSIBILIDADE
# ============================================================

NOME_TABELA_CENARIO = (
    "dim_cenario_sensibilidade"
)


inspector = inspect(
    engine
)


cenario_ja_existe = (
    inspector.has_table(
        NOME_TABELA_CENARIO
    )
)


print("=" * 70)
print("CARGA — DIM CENÁRIO SENSIBILIDADE")
print("=" * 70)


print(
    "\nTabela já existe:",
    cenario_ja_existe
)


if cenario_ja_existe:

    raise RuntimeError(
        "A tabela dim_cenario_sensibilidade já existe. "
        "Carga interrompida para evitar sobrescrita."
    )


resumo_sensibilidade.to_sql(
    name=NOME_TABELA_CENARIO,
    con=engine,
    if_exists="fail",
    index=False,
    dtype=TIPOS_SQL_CENARIO
)


print(
    "\nCarga concluída."
)

CARGA — DIM CENÁRIO SENSIBILIDADE

Tabela já existe: False

Carga concluída.


In [42]:
# ============================================================
# VALIDAÇÃO E ESTRUTURA — DIM CENÁRIO SENSIBILIDADE
# ============================================================

with engine.connect() as conexao:

    total_cenarios_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_CENARIO}`;
                """
            )
        )
        .scalar()
    )


    cenarios_distintos_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(
                    DISTINCT cenario
                )
                FROM `{NOME_TABELA_CENARIO}`;
                """
            )
        )
        .scalar()
    )


    null_cenarios_mysql = (
        conexao.execute(
            text(
                f"""
                SELECT COUNT(*)
                FROM `{NOME_TABELA_CENARIO}`
                WHERE
                    cenario IS NULL;
                """
            )
        )
        .scalar()
    )


print("=" * 70)
print("VALIDAÇÃO — DIM CENÁRIO")
print("=" * 70)


print(
    "\nLinhas DataFrame:",
    len(
        resumo_sensibilidade
    )
)


print(
    "Linhas MySQL:",
    total_cenarios_mysql
)


print(
    "\nCenários distintos:",
    cenarios_distintos_mysql
)


print(
    "NULL em cenário:",
    null_cenarios_mysql
)


# ============================================================
# PRIMARY KEY
# ============================================================

inspector = inspect(
    engine
)


pk_cenario_atual = (
    inspector
    .get_pk_constraint(
        NOME_TABELA_CENARIO
    )
)


colunas_pk_cenario = (
    pk_cenario_atual
    .get(
        "constrained_columns"
    )
    or []
)


if not colunas_pk_cenario:

    with engine.begin() as conexao:

        conexao.execute(
            text(
                f"""
                ALTER TABLE `{NOME_TABELA_CENARIO}`

                MODIFY `cenario`
                    VARCHAR(20) NOT NULL,

                ADD PRIMARY KEY (
                    `cenario`
                );
                """
            )
        )

    print(
        "\nPrimary Key criada."
    )

else:

    print(
        "\nPrimary Key já existe."
    )


# ============================================================
# AUDITORIA DA DIMENSÃO
# ============================================================

inspector = inspect(
    engine
)


pk_cenario_final = (
    inspector
    .get_pk_constraint(
        NOME_TABELA_CENARIO
    )
)


print(
    "\nPrimary Key final:"
)

print(
    pk_cenario_final
)


print(
    "\nConteúdo no MySQL:"
)


cenario_mysql = pd.read_sql(
    text(
        f"""
        SELECT *
        FROM `{NOME_TABELA_CENARIO}`
        ORDER BY
            CASE cenario
                WHEN 'base' THEN 1
                WHEN 'produtivo' THEN 2
                WHEN 'ambiental' THEN 3
                WHEN 'climatico' THEN 4
                ELSE 5
            END;
        """
    ),
    engine
)


display(
    cenario_mysql
)


# ============================================================
# INVENTÁRIO EXATO DAS 4 TABELAS
# ============================================================

inventario_final_mysql = pd.read_sql(
    text(
        """
        SELECT
            'fato_agroambiental_anual' AS tabela,
            COUNT(*) AS registros
        FROM fato_agroambiental_anual

        UNION ALL

        SELECT
            'mart_priorizacao_municipal',
            COUNT(*)
        FROM mart_priorizacao_municipal

        UNION ALL

        SELECT
            'fato_sensibilidade_priorizacao',
            COUNT(*)
        FROM fato_sensibilidade_priorizacao

        UNION ALL

        SELECT
            'dim_cenario_sensibilidade',
            COUNT(*)
        FROM dim_cenario_sensibilidade;
        """
    ),
    engine
)


print(
    "\nInventário final:"
)


display(
    inventario_final_mysql
)

VALIDAÇÃO — DIM CENÁRIO

Linhas DataFrame: 4
Linhas MySQL: 4

Cenários distintos: 4
NULL em cenário: 0

Primary Key criada.

Primary Key final:
{'constrained_columns': ['cenario'], 'name': None}

Conteúdo no MySQL:


,cenario,estrategicos_cenario,estrategicos_em_comum_com_base,retencao_dos_225_base_pct,jaccard_pct,spearman_relevancia_vs_base,spearman_pressao_vs_base
0,base,225,225,100.000000,100.000000,1.000000,1.000000
1,produtivo,250,216,96.000000,83.397683,0.985597,1.000000
2,ambiental,242,191,84.888889,69.202899,1.000000,0.917803
3,climatico,247,207,92.000000,78.113208,1.000000,0.949328



Inventário final:


,tabela,registros
0,fato_agroambiental_anual,8674
1,mart_priorizacao_municipal,1505
2,fato_sensibilidade_priorizacao,1400
3,dim_cenario_sensibilidade,4
